### apriori - use to analyze multiple categorical variables

Usually used to analyze datasets with 3 or more categorical variables, and to see if there are any combinations of categories that are underrepresented.

In [1]:
import pandas as pd

# pip install mlxtend
from mlxtend.frequent_patterns import apriori

# load the data
df = pd.read_csv("lifestyle_sustainability_data.csv")

# drop the id -column
df = df.drop("ParticipantID", axis=1)
df.head()

,Age,Location,DietType,LocalFoodFrequency,TransportationMode,EnergySource,HomeType,HomeSize,ClothingFrequency,SustainableBrands,EnvironmentalAwareness,CommunityInvolvement,MonthlyElectricityConsumption,MonthlyWaterConsumption,Gender,UsingPlasticProducts,DisposalMethods,PhysicalActivities,Rating
0,35,Urban,Mostly Plant-Based,Often,Bike,Renewable,Apartment,800,Rarely,True,5,High,100,1500,Female,Rarely,Composting,High,5
1,28,Suburban,Balanced,Sometimes,Public Transit,Mixed,House,1500,Sometimes,True,4,Moderate,250,3000,Male,Sometimes,Recycling,Moderate,4
2,65,Rural,Mostly Animal-Based,Rarely,Car,Non-Renewable,House,2500,Often,False,2,Low,400,4500,Male,Often,Landfill,Low,1
3,42,Urban,Mostly Plant-Based,Often,Walk,Renewable,Apartment,950,Sometimes,True,4,Moderate,150,2000,Female,Rarely,Recycling,High,5
4,31,Suburban,Balanced,Sometimes,Public Transit,Mixed,House,1800,Often,True,3,Low,300,3500,Non-Binary,Sometimes,Combination,Moderate,3


In [2]:
df.columns

Index(['Age', 'Location', 'DietType', 'LocalFoodFrequency',
       'TransportationMode', 'EnergySource', 'HomeType', 'HomeSize',
       'ClothingFrequency', 'SustainableBrands', 'EnvironmentalAwareness',
       'CommunityInvolvement', 'MonthlyElectricityConsumption',
       'MonthlyWaterConsumption', 'Gender', 'UsingPlasticProducts',
       'DisposalMethods', 'PhysicalActivities', 'Rating'],
      dtype='object')

### Handle categoricals as needed (there's many categorical variables in this dataset)

In [3]:
# this just converts the value of column to 0 or 1
# factorize in pandas works too, but only one column at a time

# SUSTAINABLE BRANDS => True/False => Binary category => use LabelEncoder

from sklearn.preprocessing import LabelEncoder
variables = ["SustainableBrands"]
encoder = LabelEncoder()
df[variables] = df[variables].apply(encoder.fit_transform)

In [4]:
# we have to map out the types of our categories

# Rating and EnvironmentAwareness are already in 1-5 format (ordinal category), they are good to go
# type 1: binary categories, we have: SustainableBrands => convert False to 0, True to 1
# type 2: ordinal categories, we have: LocalFoodFrequency, ClothingFrequency, CommunityInvolvement, UsingPlasticProducts, PhysicalActivities
# for ordinal categories => use map/replace -feature in pandas
# type 3: nominal categories => use OneHotEncoder: Location, DietType, TransportationMod, EnergySource, HomeType, Gender, DisposalMethods

**We have two kinds of ordinal variables here -> three options or four options**

In [5]:
 # this example is using the housing data from Moodle
varlist_three = ["CommunityInvolvement", "PhysicalActivities"]
varlist_four1 = ["LocalFoodFrequency", "ClothingFrequency"]
varlist_four2 = ["UsingPlasticProducts"]

# Defining the map function
def binary_map_three(x):
    return x.map({'Low': 0, "Moderate": 1, "High": 2})

# Defining the map function
def binary_map_four_version1(x):
    return x.map({'Rarely': 0, "Sometimes": 1, "Often": 2, "Always": 3})

# Defining the map function
def binary_map_four_version2(x):
    return x.map({'Never': 0, "Rarely": 1, "Sometimes": 2, "Often": 3})

# Applying the function to the housing list
df[varlist_three] = df[varlist_three].apply(binary_map_three)
df[varlist_four1] = df[varlist_four1].apply(binary_map_four_version1)
df[varlist_four2] = df[varlist_four2].apply(binary_map_four_version2)

In [6]:
df.head()

,Age,Location,DietType,LocalFoodFrequency,TransportationMode,EnergySource,HomeType,HomeSize,ClothingFrequency,SustainableBrands,EnvironmentalAwareness,CommunityInvolvement,MonthlyElectricityConsumption,MonthlyWaterConsumption,Gender,UsingPlasticProducts,DisposalMethods,PhysicalActivities,Rating
0,35,Urban,Mostly Plant-Based,2,Bike,Renewable,Apartment,800,0,1,5,2.0,100,1500,Female,1,Composting,2.0,5
1,28,Suburban,Balanced,1,Public Transit,Mixed,House,1500,1,1,4,1.0,250,3000,Male,2,Recycling,1.0,4
2,65,Rural,Mostly Animal-Based,0,Car,Non-Renewable,House,2500,2,0,2,0.0,400,4500,Male,3,Landfill,0.0,1
3,42,Urban,Mostly Plant-Based,2,Walk,Renewable,Apartment,950,1,1,4,1.0,150,2000,Female,1,Recycling,2.0,5
4,31,Suburban,Balanced,1,Public Transit,Mixed,House,1800,2,1,3,0.0,300,3500,Non-Binary,2,Combination,1.0,3


**The rest of the categories are nominal ... we have to use onehot-encoding**

In [7]:
 # this makes multiple columns with the variable (Separate for yes/no)
from sklearn.preprocessing import OneHotEncoder
variables = ["Location", "DietType", "TransportationMode", "EnergySource", "HomeType", "Gender", "DisposalMethods"]

# use encoder
encoder = OneHotEncoder(sparse_output=False).set_output(transform="pandas")
one_hot_encoded = encoder.fit_transform(df[variables]).astype(int)
df = pd.concat([df,one_hot_encoded],axis=1).drop(columns=variables)

In [8]:
df.head()

,Age,LocalFoodFrequency,HomeSize,ClothingFrequency,SustainableBrands,EnvironmentalAwareness,CommunityInvolvement,MonthlyElectricityConsumption,MonthlyWaterConsumption,UsingPlasticProducts,...,HomeType_House,HomeType_Other,Gender_Female,Gender_Male,Gender_Non-Binary,Gender_Prefer not to say,DisposalMethods_Combination,DisposalMethods_Composting,DisposalMethods_Landfill,DisposalMethods_Recycling
0,35,2,800,0,1,5,2.0,100,1500,1,...,0,0,1,0,0,0,0,1,0,0
1,28,1,1500,1,1,4,1.0,250,3000,2,...,1,0,0,1,0,0,0,0,0,1
2,65,0,2500,2,0,2,0.0,400,4500,3,...,1,0,0,1,0,0,0,0,1,0
3,42,2,950,1,1,4,1.0,150,2000,1,...,0,0,1,0,0,0,0,0,0,1
4,31,1,1800,2,1,3,0.0,300,3500,2,...,1,0,0,0,1,0,1,0,0,0


In [9]:
# we can always remove one of the onehotencoded variables, 
# remove: 'DietType_Mostly Plant-Based',  'TransportationMode_Walk',  'EnergySource_Renewable', 'HomeType_Other', 'Gender_Prefer not to say', 'DisposalMethods_Recycling'
removables = ['DietType_Mostly Plant-Based',  'TransportationMode_Walk',  'EnergySource_Renewable', 'HomeType_Other', 'Gender_Prefer not to say', 'DisposalMethods_Recycling']
df = df.drop(removables, axis=1)

In [10]:
df.head()

,Age,LocalFoodFrequency,HomeSize,ClothingFrequency,SustainableBrands,EnvironmentalAwareness,CommunityInvolvement,MonthlyElectricityConsumption,MonthlyWaterConsumption,UsingPlasticProducts,...,EnergySource_Mixed,EnergySource_Non-Renewable,HomeType_Apartment,HomeType_House,Gender_Female,Gender_Male,Gender_Non-Binary,DisposalMethods_Combination,DisposalMethods_Composting,DisposalMethods_Landfill
0,35,2,800,0,1,5,2.0,100,1500,1,...,0,0,1,0,1,0,0,0,1,0
1,28,1,1500,1,1,4,1.0,250,3000,2,...,1,0,0,1,0,1,0,0,0,0
2,65,0,2500,2,0,2,0.0,400,4500,3,...,0,1,0,1,0,1,0,0,0,1
3,42,2,950,1,1,4,1.0,150,2000,1,...,0,0,1,0,1,0,0,0,0,0
4,31,1,1800,2,1,3,0.0,300,3500,2,...,1,0,0,1,0,0,1,1,0,0


## Save this processed version to another csv-file

In [11]:
df.to_csv("lifestyle_sustainability_data_processed.csv", index=None)